# Jarvis-Jr lightbulb command models (train with Google Colab)

Train two **microWakeWord v2 INT8 streaming** models for the ESP32-S3 firmware:

| Command | File to download |
|---|---|
| light on | `light_on.tflite` |
| light off | `light_off.tflite` |

Use **Google Colab with a GPU**.

## Firmware constraints (must match)

The board already has the listen-window firmware. Models must be:

- microWakeWord v2 streaming quantized (`stream_state_internal_quant.tflite` class)
- input `int8`, shape `1 x stride x 40` (stride 1–16)
- output `uint8`
- 16 kHz, 40-feature / 10 ms frontend (this notebook uses that)

Drop the two files into `models/` on the firmware PC. CMake embeds them automatically.

## Colab steps

1. Runtime -> Change runtime type -> **T4 GPU** (or better). High-RAM if you have Colab Pro.
2. Run the first install cell.
3. After the install cell finishes: **Runtime -> Restart session**, then run from the **Config** cell downward (or Run all again). If the runtime times out later, re-run Config -> Drive -> Piper -> helpers; skip the big downloads if those folders already exist.
4. First-model audio playback: confirm it sounds like "light on".
5. Walk away. Data download + two training runs often take **1–3 hours** on a T4.
6. Download `light_on.tflite` and `light_off.tflite` when the last cell offers them (also copied to Drive if you mounted it).

This follows [OHF-Voice/micro-wake-word](https://github.com/OHF-Voice/micro-wake-word) `basic_training_notebook.ipynb`, except sample generation uses **Piper TTS** (`piper-tts` Python API + a LibriTTS ONNX voice). Do not install `piper-phonemize-cross` and do not run `python3 -m piper_sample_generator`. The first pass may false-trigger or miss; you can raise `--max-samples` and `training_steps` and run again.


In [ ]:
# Install microWakeWord. Restart the Colab session when this cell finishes.
# Safe to re-run after a timeout if /content/microWakeWord is already there.
import os
import platform

if platform.system() == "Darwin":
    !pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

!pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

if not os.path.isdir("./microWakeWord"):
    !git clone --depth 1 https://github.com/kahrendt/microWakeWord
!pip install -e ./microWakeWord

print("Restart the session now: Runtime -> Restart session")
print("Then run from the Config cell downward.")
print("After an idle timeout: Config -> Drive -> Piper -> helpers. Skip augmentation if wav dirs exist.")


## Config

After the restart, run this cell and everything below. Do not re-run the first install cell unless `import microwakeword` fails.

If Colab **timed out / reconnected**, packages are gone. Re-run **Config → Drive → Piper → helpers**. Skip augmentation and negatives if `mit_rirs`, `audioset_16k`, `fma_16k`, and `negative_datasets` already have files. If `import microwakeword` fails, re-run the first install cell, restart, then Config downward.


In [ ]:
# After Runtime -> Restart session, run this cell first.
!pip install -q "fsspec[http]==2025.3.0" "gcsfs==2025.3.0" "datasets>=3,<4" huggingface_hub scipy tqdm pyyaml mmap-ninja

import datasets
import tensorflow as tf

print("datasets", datasets.__version__)
if int(datasets.__version__.split(".", 1)[0]) >= 4:
    raise RuntimeError("datasets 4.x is still loaded. Runtime -> Restart session, then run this cell again.")

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow GPUs:", gpus)
if not gpus:
    print("No GPU visible. Runtime -> Change runtime type -> T4 GPU, then Restart session.")

# Phrases the firmware listens for after "Hey Jarvis".
# `texts` are Piper TTS strings (phonetic spellings often sound better).
# `confusables` are extra negatives so "light on" and "light off" do not steal each other.

PHRASES = [
    {
        "name": "light_on",
        "texts": ["light on", "lite on"],
        "confusables": ["light off", "lite off", "lights off", "turn off", "light"],
        "max_samples": 4000,
        "confusable_samples": 1500,
    },
    {
        "name": "light_off",
        "texts": ["light off", "lite off"],
        "confusables": ["light on", "lite on", "lights on", "turn on", "light"],
        "max_samples": 4000,
        "confusable_samples": 1500,
    },
]

TRAINING_STEPS = 10000
BATCH_SIZE = 128

print("Will train:", [p["name"] for p in PHRASES])
print("Piper clips use CPU. Training uses TensorFlow on the GPU listed above.")


In [ ]:
# Save finished .tflite files to Google Drive so a Colab timeout does not lose them.
try:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/jarvis-jr-models"
    import os

    os.makedirs(DRIVE_DIR, exist_ok=True)
    print("Drive:", DRIVE_DIR)
except Exception as e:
    DRIVE_DIR = None
    print("Drive not mounted:", e)


In [ ]:
# Piper TTS via Python API.
import itertools
import os
import shutil
import wave

!pip install -q "piper-tts==1.3.0"

from piper import PiperVoice, SynthesisConfig

os.makedirs("piper_models", exist_ok=True)
PIPER_MODEL = "piper_models/en_US-libritts_r-medium.onnx"
HF_VOICE = "en/en_US/libritts_r/medium/en_US-libritts_r-medium"

if not os.path.exists(PIPER_MODEL):
    from huggingface_hub import hf_hub_download

    for suffix in (".onnx", ".onnx.json"):
        cached = hf_hub_download(
            repo_id="rhasspy/piper-voices",
            filename=HF_VOICE + suffix,
        )
        dest = "piper_models/" + os.path.basename(cached)
        if not os.path.exists(dest):
            shutil.copy(cached, dest)

PIPER_VOICE = PiperVoice.load(PIPER_MODEL)
PIPER_MAX_SPEAKERS = min(200, PIPER_VOICE.config.num_speakers)
print("piper ready:", PIPER_MODEL, "speakers", PIPER_MAX_SPEAKERS)


def generate_wavs(texts, max_samples, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    per = max(1, max_samples // max(len(texts), 1))
    settings = itertools.cycle(
        itertools.product(
            range(PIPER_MAX_SPEAKERS),
            (0.75, 1.0, 1.25, 1.4),
            (0.667, 0.85, 1.0),
            (0.8,),
        )
    )
    n = 0
    for text in texts:
        for _ in range(per):
            speaker_id, length_scale, noise_scale, noise_w = next(settings)
            path = os.path.join(out_dir, f"{n}.wav")
            with wave.open(path, "wb") as wav_file:
                PIPER_VOICE.synthesize_wav(
                    text,
                    wav_file,
                    syn_config=SynthesisConfig(
                        speaker_id=speaker_id,
                        length_scale=length_scale,
                        noise_scale=noise_scale,
                        noise_w_scale=noise_w,
                    ),
                )
            n += 1
    print("wrote", n, "wavs to", out_dir)


In [ ]:
# Play one clip per phrase. If it does not sound like the command, edit PHRASES texts and re-run this cell.
from IPython.display import Audio, display

!rm -rf preview_samples
for phrase in PHRASES:
    out = f"preview_samples/{phrase['name']}"
    generate_wavs([phrase["texts"][0]], 1, out)
    wav = f"{out}/0.wav"
    print(phrase["name"], phrase["texts"][0], wav)
    display(Audio(wav, autoplay=False))


## Shared datasets (once per Colab runtime)

RIR + background noise + Hugging Face negative spectrograms. Slow, but reused for both models.

The AudioSet / FMA clips mixed into training have **mixed licenses**. Treat custom models as **non-commercial personal use** unless you replace those sets.


In [ ]:
# Downloads augmentation audio. Skip if mit_rirs / audioset_16k / fma_16k already have wavs.
import os
import shutil
from math import gcd
from pathlib import Path

import datasets
import numpy as np
import scipy
from scipy.signal import resample_poly
from tqdm import tqdm


def need_wav_dir(path):
    p = Path(path)
    if p.is_dir() and any(p.rglob("*.wav")):
        return False
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True)
    return True


def to_16k_mono_int16(arr, sr):
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim == 2:
        arr = arr.mean(axis=0 if arr.shape[0] <= 8 else 1)
    arr = np.squeeze(arr)
    sr = int(sr)
    if sr != 16000:
        g = gcd(sr, 16000)
        arr = resample_poly(arr, 16000 // g, sr // g).astype(np.float32)
    return np.clip(arr * 32767.0, -32768, 32767).astype(np.int16)


def decode_audio(audio):
    if isinstance(audio, dict):
        return np.asarray(audio["array"], dtype=np.float32), int(
            audio.get("sampling_rate") or 16000
        ), audio.get("path")
    samples = audio.get_all_samples()
    arr = np.asarray(samples.data, dtype=np.float32)
    path = getattr(audio, "path", None) or getattr(audio, "_path", None)
    return arr, int(samples.sample_rate), path


def write_decoded(output_dir, audio, fallback_name):
    arr, sr, path = decode_audio(audio)
    name = os.path.basename(path) if path else fallback_name
    stem, ext = os.path.splitext(name)
    if ext.lower() != ".wav":
        name = stem + ".wav"
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, to_16k_mono_int16(arr, sr))


output_dir = "./mit_rirs"
if need_wav_dir(output_dir):
    rir_dataset = datasets.load_dataset(
        "davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True
    )
    for i, row in enumerate(tqdm(rir_dataset)):
        write_decoded(output_dir, row["audio"], f"{i:05d}.wav")

output_dir = "./audioset_16k"
if need_wav_dir(output_dir):
    try:
        as_ds = datasets.load_dataset(
            "agkphysics/AudioSet", "balanced", split="train", streaming=True
        )
        for i, row in enumerate(tqdm(as_ds, total=600, desc="audioset")):
            if i >= 600:
                break
            write_decoded(output_dir, row["audio"], f"{i:05d}.wav")
        if not any(Path(output_dir).rglob("*.wav")):
            raise RuntimeError("no AudioSet wavs written")
    except Exception as e:
        print("AudioSet skipped (FMA noise is enough):", e)
        shutil.rmtree(output_dir, ignore_errors=True)

output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    fname = "fma_xs.zip"
    !wget -O fma/{fname} https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/{fname}
    !cd fma && unzip -q {fname}

output_dir = "./fma_16k"
if need_wav_dir(output_dir):
    mp3s = [str(i) for i in Path("fma/fma_small").glob("**/*.mp3")]
    fma_dataset = datasets.Dataset.from_dict({"audio": mp3s})
    fma_dataset = fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for src, row in zip(mp3s, tqdm(fma_dataset)):
        write_decoded(output_dir, row["audio"], Path(src).stem + ".wav")

print("augmentation audio ready")


In [ ]:
# Pre-generated negative spectrograms for microWakeWord (Hugging Face).
import os

output_dir = "./negative_datasets"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    link_root = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
    for fname in ["dinner_party.zip", "dinner_party_eval.zip", "no_speech.zip", "speech.zip"]:
        !wget -O negative_datasets/{fname} {link_root}{fname}
        !unzip -q negative_datasets/{fname} -d {output_dir}

print("negatives ready")


In [ ]:
# Train one phrase -> /content/<name>.tflite (streaming INT8).
import os
import shutil
from pathlib import Path

!pip install -q "fsspec[http]==2025.3.0" "gcsfs==2025.3.0" "datasets>=3,<4" huggingface_hub scipy tqdm pyyaml mmap-ninja

if not os.path.isdir("./microWakeWord"):
    raise RuntimeError(
        "microWakeWord is missing (runtime was recycled). "
        "Re-run the first install cell, Restart session, then Config → Drive → Piper → this cell. "
        "Skip augmentation/negatives if those folders already have files."
    )
!pip install -q -e ./microWakeWord

if "PHRASES" not in globals():
    raise RuntimeError("Config is not loaded. Re-run the Config cell, then this cell.")
if "generate_wavs" not in globals():
    raise RuntimeError("Piper is not loaded. Re-run the Piper cell, then this cell.")
if "DRIVE_DIR" not in globals():
    DRIVE_DIR = None

import yaml
from mmap_ninja.ragged import RaggedMmap
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration


def write_features(wav_dir, feat_dir, slide_train=10):
    if os.path.exists(feat_dir):
        shutil.rmtree(feat_dir)
    os.makedirs(feat_dir, exist_ok=True)
    clips = Clips(
        input_directory=wav_dir,
        file_pattern="*.wav",
        max_clip_duration_s=None,
        remove_silence=False,
        random_split_seed=10,
        split_count=0.1,
    )
    augmenter = Augmentation(
        augmentation_duration_s=3.2,
        augmentation_probabilities={
            "SevenBandParametricEQ": 0.1,
            "TanhDistortion": 0.1,
            "PitchShift": 0.1,
            "BandStopFilter": 0.1,
            "AddColorNoise": 0.1,
            "AddBackgroundNoise": 0.75,
            "Gain": 1.0,
            "RIR": 0.5,
        },
        impulse_paths=[p for p in ["mit_rirs"] if Path(p).is_dir() and any(Path(p).rglob("*.wav"))],
        background_paths=[
            p
            for p in ["fma_16k", "audioset_16k"]
            if Path(p).is_dir() and any(Path(p).rglob("*.wav"))
        ],
        background_min_snr_db=-5,
        background_max_snr_db=10,
        min_jitter_s=0.195,
        max_jitter_s=0.205,
    )
    for split in ["training", "validation", "testing"]:
        out_dir = os.path.join(feat_dir, split)
        os.makedirs(out_dir, exist_ok=True)
        split_name = "train"
        repetition = 2
        spectrograms = SpectrogramGeneration(
            clips=clips, augmenter=augmenter, slide_frames=slide_train, step_ms=10
        )
        if split == "validation":
            split_name = "validation"
            repetition = 1
        elif split == "testing":
            split_name = "test"
            repetition = 1
            spectrograms = SpectrogramGeneration(
                clips=clips, augmenter=augmenter, slide_frames=1, step_ms=10
            )
        RaggedMmap.from_generator(
            out_dir=os.path.join(out_dir, "wakeword_mmap"),
            sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
            batch_size=100,
            verbose=True,
        )


def write_yaml(train_dir, positive_dir, confusable_dir):
    config = {
        "window_step_ms": 10,
        "train_dir": train_dir,
        "features": [
            {
                "features_dir": positive_dir,
                "sampling_weight": 2.0,
                "penalty_weight": 1.0,
                "truth": True,
                "truncation_strategy": "truncate_start",
                "type": "mmap",
            },
            {
                "features_dir": confusable_dir,
                "sampling_weight": 8.0,
                "penalty_weight": 2.0,
                "truth": False,
                "truncation_strategy": "random",
                "type": "mmap",
            },
            {
                "features_dir": "negative_datasets/speech",
                "sampling_weight": 10.0,
                "penalty_weight": 1.0,
                "truth": False,
                "truncation_strategy": "random",
                "type": "mmap",
            },
            {
                "features_dir": "negative_datasets/dinner_party",
                "sampling_weight": 10.0,
                "penalty_weight": 1.0,
                "truth": False,
                "truncation_strategy": "random",
                "type": "mmap",
            },
            {
                "features_dir": "negative_datasets/no_speech",
                "sampling_weight": 5.0,
                "penalty_weight": 1.0,
                "truth": False,
                "truncation_strategy": "random",
                "type": "mmap",
            },
        ],
        "training_steps": [TRAINING_STEPS],
        "positive_class_weight": [1],
        "negative_class_weight": [20],
        "learning_rates": [0.001],
        "batch_size": BATCH_SIZE,
        "time_mask_max_size": [0],
        "time_mask_count": [0],
        "freq_mask_max_size": [0],
        "freq_mask_count": [0],
        "eval_step_interval": 2500,
        "clip_duration_ms": 1500,
        "target_minimization": 0.9,
        "minimization_metric": None,
        "maximization_metric": "accuracy",
    }
    with open("training_parameters.yaml", "w") as f:
        yaml.dump(config, f)


def _wav_count(path):
    p = Path(path)
    return len(list(p.glob("*.wav"))) if p.is_dir() else 0


def _has_features(feat_dir):
    return Path(feat_dir, "training", "wakeword_mmap").is_dir()


def train_phrase(phrase):
    name = phrase["name"]
    print("====", name, "====")
    pos_wav = f"work/{name}/pos_wav"
    conf_wav = f"work/{name}/conf_wav"
    pos_feat = f"work/{name}/pos_feat"
    conf_feat = f"work/{name}/conf_feat"
    train_dir = f"trained_models/{name}"
    src = os.path.join(
        train_dir, "tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"
    )
    dst = f"/content/{name}.tflite"

    if _wav_count(pos_wav) < phrase["max_samples"]:
        generate_wavs(phrase["texts"], phrase["max_samples"], pos_wav)
    else:
        print("reuse", pos_wav, _wav_count(pos_wav), "wavs")
    if _wav_count(conf_wav) < phrase["confusable_samples"]:
        generate_wavs(phrase["confusables"], phrase["confusable_samples"], conf_wav)
    else:
        print("reuse", conf_wav, _wav_count(conf_wav), "wavs")
    if not _has_features(pos_feat):
        write_features(pos_wav, pos_feat)
    else:
        print("reuse", pos_feat)
    if not _has_features(conf_feat):
        write_features(conf_wav, conf_feat, slide_train=1)
    else:
        print("reuse", conf_feat)
    write_yaml(train_dir, pos_feat, conf_feat)

    if not os.path.isfile(src):
        !python -m microwakeword.model_train_eval \
            --training_config='training_parameters.yaml' \
            --train 1 \
            --restore_checkpoint 1 \
            --test_tf_nonstreaming 0 \
            --test_tflite_nonstreaming 0 \
            --test_tflite_nonstreaming_quantized 0 \
            --test_tflite_streaming 0 \
            --test_tflite_streaming_quantized 1 \
            --use_weights "best_weights" \
            mixednet \
            --pointwise_filters "64,64,64,64" \
            --repeat_in_block "1, 1, 1, 1" \
            --mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
            --residual_connection "0,0,0,0" \
            --first_conv_filters 32 \
            --first_conv_kernel_size 5 \
            --stride 3

    if not os.path.isfile(src):
        raise FileNotFoundError(
            src + " - training was interrupted before export. "
            "Re-run this train loop; wavs/features are reused. "
            "Do not stop at the Step # eval RAM warning."
        )
    shutil.copy2(src, dst)
    if DRIVE_DIR:
        shutil.copy2(src, os.path.join(DRIVE_DIR, f"{name}.tflite"))
    print("wrote", dst, "bytes", os.path.getsize(dst))
    return dst


print("helpers ready")


In [ ]:
# Long cell. Do not interrupt after "Step #…" — the CPU RAM warning is normal.
# Re-run after a timeout: helpers, then this cell. Existing wavs/features are reused.
outputs = []
for phrase in PHRASES:
    outputs.append(train_phrase(phrase))
print("done:", outputs)


In [ ]:
# Download both models to PC. Copy them to Jarvis-Jr/models/ then rebuild firmware.
from pathlib import Path

try:
    from google.colab import files

    for name in ("light_on", "light_off"):
        path = f"/content/{name}.tflite"
        assert Path(path).is_file(), path
        files.download(path)
except Exception as e:
    print("files.download skipped:", e)
    print("Copy /content/light_on.tflite and /content/light_off.tflite yourself.")
